<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 강화학습·에이전트 · 09. AlphaZero / MCTS

## Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm

- **원 논문:** [Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm](https://arxiv.org/abs/1712.01815)
- **저자 / 발표:** David Silver et al. · arXiv / Science (2017)
- **난이도 / 예상 시간:** 고급 · 약 75분
- **선수 지식:** MCTS, policy/value network, self-play targets
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 로컬 deterministic MDP에서 PUCT visit policy와 joint policy/value loss를 구현한다.

**축소하거나 생략한 부분:** 논문은 chess/shogi self-play와 대규모 residual network를 사용한다. 축소판은 exact model과 value-iteration leaf oracle로 MCTS/visit target/loss를 분리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Main text, network paragraph and Eq. (1) | (p,v)=f_theta(s) and joint loss | visit policy cross-entropy, outcome value MSE, L2를 결합한다. |
| Main text, MCTS paragraph | PUCT search and root visit counts | prior P, mean Q, visit N으로 action을 고르고 pi를 만든다. |
| Main text, self-play paragraph | search-improved policy targets | a_t~pi_t와 terminal z의 축소판으로 모든 state search targets를 만든다. |

## 핵심 재현

        네트워크 $(p,v)=f_\theta(s)$가 prior와 leaf value를 내고 MCTS root visit counts가
        개선된 policy $\pi$를 만듭니다. 마지막에는 Eq. (1)의 policy/value/L2 loss로 network를 학습합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(491)
np.random.seed(491)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
\mathcal L=(z-v)^2-\pi^\top\log p+c\lVert\theta\rVert_2^2,\qquad U(s,a)=c_{puct}P(s,a)\frac{\sqrt{\sum_bN(s,b)}}{1+N(s,a)}
$$

- **기호와 역할:** $p/v$는 network policy/value, $\pi$는 MCTS visit target, $P,Q,N$은 tree prior/mean value/visit count입니다.
- **shape/state 계약:** `state [B] → policy logits [B, A], value [B]; root P/Q/N/pi [A]`
- **논문 위치:** `Main text, network paragraph and Eq. (1)`의 **(p,v)=f_theta(s) and joint loss**
- **코드 Task:** PUCT selection, visit policy, policy-value-L2 loss와 search target을 구현합니다.
- **학습·평가 흐름:** network prior/value → MCTS simulations → visit pi → joint update → agreement 평가
- **원문 대비 한계:** 논문은 chess/shogi self-play와 대규모 residual network를 사용한다. 축소판은 exact model과 value-iteration leaf oracle로 MCTS/visit target/loss를 분리한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class AlphaZeroSearchAgent(nn.Module):
    """AlphaZero / MCTS의 핵심 학습·평가 경로. 입출력 계약: state [B] → policy logits [B, A],
    value [B]; root P/Q/N/pi [A]."""

    def __init__(
        self,
        policy_value_network,
        optimizer,
        transition_fn,
        leaf_action_values,
        discount,
        action_count,
        exploration=1.5,
    ):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        raise NotImplementedError("TODO: __init__ 본문을 구현하세요.")

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        raise NotImplementedError("TODO: config 본문을 구현하세요.")

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: state [B] → policy logits [B, A], value
        [B]; root P/Q/N/pi [A]."""
        raise NotImplementedError("TODO: forward 본문을 구현하세요.")

    def select_action(self, state, simulations=80, temperature=1.0):
        """PUCT simulation을 수행하고 temperature visit policy, count, prior와 행동을 반환한다."""
        raise NotImplementedError("TODO: select_action 본문을 구현하세요.")

    def loss(self, state, target_policy, target_value):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        raise NotImplementedError("TODO: loss 본문을 구현하세요.")

    def update(self, state, target_policy, target_value):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        raise NotImplementedError("TODO: update 본문을 구현하세요.")

    def evaluate(self, state, target_policy, target_value, visit_count, raw_prior):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        raise NotImplementedError("TODO: evaluate 본문을 구현하세요.")

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `Main text, MCTS paragraph` — **PUCT search and root visit counts**
- **구현 이유:** prior P, mean Q, visit N으로 action을 고르고 pi를 만든다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class PVNet(nn.Module):
    """PVNet 연습 skeleton. 공개 method의 signature와 반환 계약은 정답과 같다."""

    def __init__(self):
        """정답과 동일한 공개 signature를 유지한다. shape 계약: state [B] → policy logits [B, A],
        value [B]; root P/Q/N/pi [A]."""
        raise NotImplementedError("TODO: __init__ 본문을 구현하세요.")

    def forward(self, state):
        """정답과 동일한 공개 signature를 유지한다. shape 계약: state [B] → policy logits [B, A],
        value [B]; root P/Q/N/pi [A]."""
        raise NotImplementedError("TODO: forward 본문을 구현하세요.")


# TODO: policy/value network와 Agent를 만들고 select_action으로 root search를
# 실행하세요. 반환된 visit count와 temperature policy를 검증하세요.
raise NotImplementedError

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `Main text, self-play paragraph` — **search-improved policy targets**
- **구현 이유:** a_t~pi_t와 terminal z의 축소판으로 모든 state search targets를 만든다.
- **읽을 코드:** 아래 `search` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
# TODO: 모든 nonterminal state에서 Agent.select_action을 호출해 improved target_pi,
# visit_counts, raw_priors를 구성하세요.
raise NotImplementedError

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `Main text, network paragraph and Eq. (1)` — **(p,v)=f_theta(s) and joint loss**
- **구현 이유:** visit policy cross-entropy, outcome value MSE, L2를 결합한다.
- **읽을 코드:** 아래 `training-evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
# TODO: 모든 nonterminal state의 search pi와 z=max Q*로 Eq. (1) network를 200회 학습하고
# 시각화하세요.

# 통합 완료 조건: AlphaZeroSearchAgent.update(...)를 반복하고,
# history, parameter_delta, 유한 loss를 검증합니다.
raise NotImplementedError

왜 raw network policy p 대신 MCTS visit policy pi를 target으로 쓰나요?

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `Main text, network paragraph and Eq. (1)`의 **(p,v)=f_theta(s) and joint loss**
- **핵심 식:**

  $$
  \mathcal L=(z-v)^2-\pi^\top\log p+c\lVert\theta\rVert_2^2,\qquad U(s,a)=c_{puct}P(s,a)\frac{\sqrt{\sum_bN(s,b)}}{1+N(s,a)}
  $$

- **입출력 shape:** `state [B] → policy logits [B, A], value [B]; root P/Q/N/pi [A]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `AlphaZeroSearchAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. visit policy cross-entropy, outcome value MSE, L2를 결합한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
# TODO 5: AlphaZeroSearchAgent 기반 학습의 최종 증거를 검증하세요.
# 계약: history, portfolio_metrics, parameter_delta를 모두 검사합니다.
raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.